# Step 8b - Gender-focused annotation schema, Layer A (linguistic annotation)

**Status.** Replaces the Step 8 field set for the primary gender model. The Step 8 notebook and its outputs are kept unchanged for the record.

**Why.** The Step 8 field `bias_type` mixed three questions: is gender mentioned, does the wording impose a restriction or stereotype, and should it be rewritten. Layer A records only the **linguistic phenomenon**. It does not decide bias. Bias judgement and the rewrite decision are separate fields (Layer B), which are **not defined in this notebook** and need a project decision.

**Scope.** Primary task is gender. Age, marital status, appearance and other attributes are out of the primary model; the sample strata that selected them are kept so those lines can be reviewed later.

This notebook trains nothing and creates no labels. Outputs go to `outputs/equal_step8b_gender_schema/`.

## 1. Layer A fields

| Field | Values | Rule |
|---|---|---|
| `unit_id` | Step 2 `sentence_id` | Identifies the unit. |
| `ad_id` | Step 2 `record_id` | Parent posting. |
| `component` | `job_title`, `qualification`, `body_other` | Step 2 section. The Step 2 value `body_other` is the "body" class. |
| `gender_reference` | `yes`, `no` | `yes` only when the unit names or points to a gender category (male, female, men, women, Mr/Ms, he/she, a gendered job title such as "salesman"). Coded traits like "assertive" are `no`. |
| `gender_expression_type` | `restriction`, `preference`, `binary_listing`, `gendered_title_pronoun`, `gender_coded`, `inclusive`, `none`, `unclear` | The linguistic phenomenon: `restriction` = only one gender may apply ("Male only"); `preference` = one gender preferred; `binary_listing` = both genders listed ("Female/Male"); `gendered_title_pronoun` = gendered title or pronoun; `gender_coded` = no gender mention but stereotype-associated wording; `inclusive` = statement open to all genders; `none`; `unclear`. This is a description, not a bias verdict. |
| `gender_direction` | `male`, `female`, `multiple`, `unspecified`, `na` | The referent. `na` when `gender_reference = no`. `binary_listing` is `multiple`. |
| `gender_coding` | `agentic`, `communal`, `mixed`, `na`, `unclear` | Stereotypically coded wording. Only for `gender_coded`; use `na` otherwise. Coded wording that clearly describes the duty rather than the person is marked `context_dependent = yes`. |
| `span_text` | exact substring of the unit | The **smallest complete expression** that supports the annotation (e.g. "male applicant" in "We are looking for a strong male applicant"). One span per row. When independent issues occur, add another row with the same `unit_id` and a different span. |
| `span_start`, `span_end` | character offsets, end exclusive | Offsets into the original-case Step 2 unit text: `sentence[span_start:span_end] == span_text`. |
| `context_dependent` | `yes`, `no` | The label cannot be fixed without the ad context. |
| `language_quality_flag` | `yes`, `no` | Typo, malapropism or non-standard grammar, recorded independently. |
| `notes` | free text | |

**Consistency rules checked by the validator**
- `gender_reference = no` allows only `gender_coded`, `none`, `unclear`, and requires `gender_direction = na`.
- `gender_reference = yes` allows `restriction`, `preference`, `binary_listing`, `gendered_title_pronoun`, `inclusive`, `unclear`, and requires a direction other than `na`.
- `binary_listing` requires `gender_direction = multiple`. `gender_coded` requires `gender_coding` other than `na`; every other type requires `gender_coding = na`.
- Every type except `none` requires a span with correct offsets.

**Offsets and normalised text.** Offsets always refer to the canonical Step 2 unit text, which keeps the original casing. A lower-cased copy has identical offsets (checked below). Any normalisation that changes length (for example Step 3) needs an offset map back to the unit, built with the helper in section 3.

**Mapping from Step 8 (documentation only, no automatic conversion)**

| Step 8 value | Layer A equivalent |
|---|---|
| `bias_type = explicit` and `explicit_direction = male/female` | `gender_expression_type = restriction` or `preference` (the annotator decides) |
| `explicit_direction = both_listed` | `binary_listing`, `gender_direction = multiple` |
| `explicit_direction = title_or_pronoun` | `gendered_title_pronoun` |
| `neutral_statement` | `inclusive` |
| `gender_coding = agentic/communal` (implicit) | `gender_coded` with `agentic`/`communal`; no bias verdict is implied |
| `attribute` age, marital_status, appearance_physical, ... | outside the primary gender model (future `other_bias_review_flag`, Layer B) |

In [1]:
import json
from pathlib import Path
import pandas as pd

OUT = Path("outputs") / "equal_step8b_gender_schema"; OUT.mkdir(parents=True, exist_ok=True)

SCHEMA = {
    "component": ["job_title", "qualification", "body_other"],
    "gender_reference": ["yes", "no"],
    "gender_expression_type": ["restriction", "preference", "binary_listing", "gendered_title_pronoun", "gender_coded", "inclusive", "none", "unclear"],
    "gender_direction": ["male", "female", "multiple", "unspecified", "na"],
    "gender_coding": ["agentic", "communal", "mixed", "na", "unclear"],
    "context_dependent": ["yes", "no"],
    "language_quality_flag": ["yes", "no"],
}
REF_NO_TYPES = {"gender_coded", "none", "unclear"}
REF_YES_TYPES = {"restriction", "preference", "binary_listing", "gendered_title_pronoun", "inclusive", "unclear"}
(OUT / "step8b_layerA_schema.json").write_text(json.dumps(SCHEMA, indent=2), encoding="utf-8")

units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
old = pd.read_csv("outputs/equal_step8_annotation/step8_annotation_sample.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("Step 8 sample:", len(old), "units | Step 2 units:", len(units))
assert set(old.sentence_id) <= set(units.sentence_id)
assert (old.set_index("sentence_id").sentence == units.set_index("sentence_id").loc[old.sentence_id, "sentence"]).all()
print("component values in sample:", old.component.value_counts().to_dict())
assert set(old.component) <= set(SCHEMA["component"])

Step 8 sample: 596 units | Step 2 units: 7971
component values in sample: {'qualification': 556, 'job_title': 32, 'body_other': 8}


## 2. Annotation sample in the Layer A format

The same 596 units as Step 8 (so the earlier sampling work is reused). No label is filled in; the old label columns are dropped. The `stratum` column is a sampling stratum from rule flags, not a label, and it should be hidden from annotators if blind labelling is wanted.

In [2]:
LABELS = ["gender_reference", "gender_expression_type", "gender_direction", "gender_coding", "span_text", "span_start", "span_end",
          "context_dependent", "language_quality_flag", "notes"]
samp = old.rename(columns={"sentence_id": "unit_id", "record_id": "ad_id"})[["unit_id", "ad_id", "employer_id", "component", "sentence", "ad_context", "stratum"]].copy()
for c in LABELS: samp[c] = ""
samp.to_csv(OUT / "step8b_layerA_annotation_sample.csv", index=False, encoding="utf-8-sig")
print("rows:", len(samp), "| ads:", samp.ad_id.nunique(), "| employers:", samp.employer_id.nunique())
print(samp.stratum.value_counts().to_string())
print("columns:", samp.columns.tolist())

rows: 596 | ads: 469 | employers: 208
stratum
control               100
inclusive_wording      86
coded_feminine         80
coded_masculine        80
proxy_appearance       60
restrictive_female     60
restrictive_male       60
proxy_age              40
proxy_marital          30
columns: ['unit_id', 'ad_id', 'employer_id', 'component', 'sentence', 'ad_context', 'stratum', 'gender_reference', 'gender_expression_type', 'gender_direction', 'gender_coding', 'span_text', 'span_start', 'span_end', 'context_dependent', 'language_quality_flag', 'notes']


## 3. Span helper, offset map and validator

`locate_span` finds exact offsets for a phrase (and refuses ambiguous matches), so annotators or tools do not type offsets by hand. `lower_with_offsets` shows that lower-casing keeps offsets; a length-changing normaliser must return its own map.

In [3]:
def locate_span(sentence, phrase):
    starts = [i for i in range(len(sentence)) if sentence.startswith(phrase, i)]
    if len(starts) != 1:
        raise ValueError(f"{len(starts)} matches for {phrase!r}; give a longer phrase or set offsets by hand")
    return starts[0], starts[0] + len(phrase)

def lower_with_offsets(sentence):
    low = sentence.lower()
    if len(low) != len(sentence):
        raise ValueError("lower-casing changed length; build an explicit offset map")
    return low, list(range(len(sentence)))

def validate(df):
    """Return (row index, problem) pairs for filled rows."""
    errors = []
    f = df[df.gender_expression_type != ""]
    for i, r in f.iterrows():
        for col in ("gender_reference", "gender_expression_type", "gender_direction", "gender_coding", "context_dependent", "language_quality_flag"):
            if r[col] not in SCHEMA[col]: errors.append((i, f"{col} value {r[col]!r}"))
        t, ref, dr, gc = r.gender_expression_type, r.gender_reference, r.gender_direction, r.gender_coding
        if ref == "no" and (t not in REF_NO_TYPES or dr != "na"): errors.append((i, "reference=no needs type in gender_coded/none/unclear and direction=na"))
        if ref == "yes" and (t not in REF_YES_TYPES or dr == "na"): errors.append((i, "reference=yes needs a gender type and a direction other than na"))
        if t == "binary_listing" and dr != "multiple": errors.append((i, "binary_listing needs direction=multiple"))
        if t == "gender_coded" and gc == "na": errors.append((i, "gender_coded needs gender_coding"))
        if t != "gender_coded" and gc != "na": errors.append((i, "gender_coding must be na unless gender_coded"))
        if t != "none":
            try:
                s, e = int(r.span_start), int(r.span_end)
                if not r.span_text or r.sentence[s:e] != r.span_text: errors.append((i, "span_text does not match offsets"))
            except ValueError:
                errors.append((i, "span offsets missing"))
        elif r.span_text:
            errors.append((i, "type none must not have a span"))
    return errors, len(f)

## 4. Validator self-test (synthetic rows only; nothing is written to the annotation file)

In [4]:
def row(**kw):
    base = dict(sentence="", gender_reference="no", gender_expression_type="none", gender_direction="na", gender_coding="na",
                span_text="", span_start="", span_end="", context_dependent="no", language_quality_flag="no")
    base.update(kw); return base

s1 = "We are looking for a strong male applicant"; a, b = locate_span(s1, "male applicant")
s2 = "Female/Male"; s3 = "Must be assertive and competitive"; a3, b3 = locate_span(s3, "assertive and competitive")
good = [
    row(sentence=s1, gender_reference="yes", gender_expression_type="restriction", gender_direction="male", span_text="male applicant", span_start=str(a), span_end=str(b)),
    row(sentence=s2, gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span_text=s2, span_start="0", span_end=str(len(s2))),
    row(sentence=s3, gender_expression_type="gender_coded", gender_coding="agentic", context_dependent="yes", span_text="assertive and competitive", span_start=str(a3), span_end=str(b3)),
    row(sentence="With at least 1 year experience"),
]
bad = [
    row(sentence=s1, gender_reference="no", gender_expression_type="restriction", gender_direction="male", span_text="male applicant", span_start=str(a), span_end=str(b)),
    row(sentence=s2, gender_reference="yes", gender_expression_type="binary_listing", gender_direction="male", span_text=s2, span_start="0", span_end=str(len(s2))),
    row(sentence=s3, gender_expression_type="gender_coded", gender_coding="na", span_text="assertive", span_start="0", span_end="9"),
    row(sentence=s1, gender_reference="yes", gender_expression_type="preference", gender_direction="male", span_text="male applicant", span_start="0", span_end="5"),
]
eg, n = validate(pd.DataFrame(good)); eb, nb_ = validate(pd.DataFrame(bad))
print(f"valid rows: {n} checked, {len(eg)} errors (expected 0)")
print(f"invalid rows: {nb_} checked, {len(eb)} errors (expected 4 rows rejected):")
for i, m in eb: print("  row", i, "-", m)
assert len(eg) == 0 and {i for i, _ in eb} == {0, 1, 2, 3}
low, _ = lower_with_offsets(s1); print("lower-case keeps offsets:", low[a:b])
try: locate_span("male and male", "male")
except ValueError as e: print("ambiguous span refused:", e)

valid rows: 4 checked, 0 errors (expected 0)
invalid rows: 4 checked, 5 errors (expected 4 rows rejected):
  row 0 - reference=no needs type in gender_coded/none/unclear and direction=na
  row 1 - binary_listing needs direction=multiple
  row 2 - gender_coded needs gender_coding
  row 2 - span_text does not match offsets
  row 3 - span_text does not match offsets
lower-case keeps offsets: male applicant
ambiguous span refused: 2 matches for 'male'; give a longer phrase or set offsets by hand


## 5. What is not done here
- No human labels exist; the label columns are empty.
- Layer B (gender bias class, rewrite requirement, other-bias review flag) is not defined; the "Female/Male" decision (`binary_listing`; bias class and rewrite handled in Layer B) therefore has no bias field to land in yet.
- Step 9 silver labels, the classifier and the rewrite steps still use the Step 8 definition (explicit includes age and marital). They are not changed.
- Sampling strata come from the Step 6 rules, so rare phenomena the rules do not flag are under-sampled.